In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_I.xpt"
csv_file = DATA_DIR / "DEMO_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2015-2016 DEMO_I XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SDDSRVYR",
    "RIDSTATR",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDRETH1",
    "RIDRETH3",
    "RIDEXMON",
    "RIDEXAGM",
    "DMQMILIZ",
    "DMQADFC",
    "DMDBORN4",
    "DMDCITZN",
    "DMDYRSUS",
    "DMDEDUC3",
    "DMDEDUC2",
    "DMDMARTL",
    "RIDEXPRG",
    "SIALANG",
    "SIAPROXY",
    "SIAINTRP",
    "FIALANG",
    "FIAPROXY",
    "FIAINTRP",
    "MIALANG",
    "MIAPROXY",
    "MIAINTRP",
    "AIALANGA",
    "DMDHHSIZ",
    "DMDFMSIZ",
    "DMDHHSZA",
    "DMDHHSZB",
    "DMDHHSZE",
    "DMDHRGND",
    "DMDHRAGE",
    "DMDHRBR4",
    "DMDHREDU",
    "DMDHRMAR",
    "DMDHSEDU",
    "WTINT2YR",
    "WTMEC2YR",
    "SDMVPSU",
    "SDMVSTRA",
    "INDHHIN2",
    "INDFMIN2",
    "INDFMPIR"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DEMO_I column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9971:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 47:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,971 rows x 47 columns."
)


# ============================================================
# 4. DEFINE GENUINE DECIMAL FIELDS
# ============================================================

#
# IMPORTANT:
#
# DEMO_I is NOT an all-integer file.
#
#
# WTINT2YR
#     Full sample interview weight.
#
# WTMEC2YR
#     Full sample MEC examination weight.
#
# INDFMPIR
#     Family income-to-poverty ratio.
#
#
# These three fields must remain floating point.
#
# DO NOT:
#
#     round()
#
# DO NOT:
#
#     astype(Int64)
#
# DO NOT:
#
#     globally remove ".0"
#
# from these variables.
#

DECIMAL_COLUMNS = [
    "WTINT2YR",
    "WTMEC2YR",
    "INDFMPIR"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 3:

    raise ValueError(
        "Expected exactly 3 decimal fields."
    )


if len(INTEGER_COLUMNS) != 44:

    raise ValueError(
        "Expected exactly 44 integer/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation:
#
#     5.397605346934028e-79
#
# This corresponds to real numeric zero in specific fields.
#
# NEVER globally replace it before validating exact
# variable/count locations.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 47 DEMO_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact attached DEMO_I.xpt:
#
# RIDAGEYR      396
# RIDAGEMN       31
# RIDEXAGM       10
# DMDEDUC3      239
# DMDHHSZA    6,298
# DMDHHSZB    4,715
# DMDHHSZE    7,151
# WTMEC2YR      427
# INDFMPIR       92
#
# TOTAL      19,359
#

EXPECTED_TINY_COUNTS = {
    "RIDAGEYR": 396,
    "RIDAGEMN": 31,
    "RIDEXAGM": 10,
    "DMDEDUC3": 239,
    "DMDHHSZA": 6298,
    "DMDHHSZB": 4715,
    "DMDHHSZE": 7151,
    "WTMEC2YR": 427,
    "INDFMPIR": 92
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DEMO_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 19359:

    raise ValueError(
        f"Unexpected total tiny count: {tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before tiny-value restoration."
    )


print(
    "\nPASS: Exact 19,359-value tiny-zero "
    "pattern across 9 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Restore only the nine validated variables.
#

print(
    "\n--- VALIDATED ZERO RESTORATION ---"
)


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 19359:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

#
# expected_df represents exactly what the corrected
# source should contain.
#
# The ONLY numerical modifications are the validated
# tiny -> zero restorations.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# MUST run before Int64 conversion.
#
# np.round() is used ONLY for detecting fractional
# observations.
#
# It does NOT change any source value.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(
            values
        ),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} "
            "fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exact genuine decimal fields detected:"
)

print(
    DECIMAL_COLUMNS
)


# ============================================================
# 11. DECIMAL FIELD DETAILS
# ============================================================

print(
    "\n--- DECIMAL FIELD DETAILS ---"
)


for col in DECIMAL_COLUMNS:

    s = df[col]


    values = (
        s.dropna()
        .to_numpy(
            dtype=float
        )
    )


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(
                    values
                ),
                rtol=0,
                atol=1e-12
            )
        )
        .sum()
    )


    whole_looking_count = (
        len(values)
        -
        fractional_count
    )


    print(
        f"\n{col}"
    )

    print(
        "Nonmissing:",
        f"{int(s.notna().sum()):,}"
    )

    print(
        "Missing:",
        f"{int(s.isna().sum()):,}"
    )

    print(
        "Minimum:",
        s.min()
    )

    print(
        "Maximum:",
        s.max()
    )

    print(
        "Fractional observations:",
        f"{fractional_count:,}"
    )

    print(
        "Whole-looking observations:",
        f"{whole_looking_count:,}"
    )


# ============================================================
# 12. VALIDATE WTINT2YR
# ============================================================

#
# CDC:
#
# 3293.928267 to 233755.84185
# n = 9,971
#

if int(
    df["WTINT2YR"]
    .notna()
    .sum()
) != 9971:

    raise ValueError(
        "WTINT2YR nonmissing count differs."
    )


if df["WTINT2YR"].min() != 3293.928267:

    raise ValueError(
        "Unexpected WTINT2YR minimum."
    )


if df["WTINT2YR"].max() != 233755.84185:

    raise ValueError(
        "Unexpected WTINT2YR maximum."
    )


print(
    "\nPASS: WTINT2YR exact range validated."
)


# ============================================================
# 13. VALIDATE WTMEC2YR
# ============================================================

#
# CDC:
#
# positive MEC sample weight = 9,544 participants
#
# 0 = Not MEC examined       = 427 participants
#
#
# IMPORTANT:
#
# WTMEC2YR is still a GENUINE DECIMAL FIELD.
#
# Do NOT convert the entire variable to Int64 merely
# because 427 observations happen to be exactly zero.
#

wtmec_positive_count = int(
    df["WTMEC2YR"]
    .gt(0)
    .sum()
)


wtmec_zero_count = int(
    df["WTMEC2YR"]
    .eq(0)
    .sum()
)


if wtmec_positive_count != 9544:

    raise ValueError(
        "Expected 9,544 positive WTMEC2YR values."
    )


if wtmec_zero_count != 427:

    raise ValueError(
        "Expected 427 WTMEC2YR zeros."
    )


if int(
    df["WTMEC2YR"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Unexpected missing WTMEC2YR values."
    )


print(
    "PASS: WTMEC2YR 9,544 positive + "
    "427 zero values validated."
)


# ============================================================
# 14. WTMEC2YR / RIDSTATR ROUTING CHECK
# ============================================================

#
# CDC:
#
# RIDSTATR = 1
#     Interviewed only
#
# WTMEC2YR = 0
#     Not MEC examined
#
# These should identify EXACTLY the same 427 records.
#

interview_only = (
    df["RIDSTATR"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


zero_mec_weight = (
    df["WTMEC2YR"]
    .eq(0)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


wtmec_route_mismatch = int(
    np.sum(
        interview_only
        !=
        zero_mec_weight
    )
)


print(
    "\n--- WTMEC2YR / RIDSTATR CHECK ---"
)


print(
    "Routing mismatches:",
    wtmec_route_mismatch
)


if wtmec_route_mismatch != 0:

    raise ValueError(
        "WTMEC2YR=0 does not exactly match "
        "RIDSTATR=1."
    )


print(
    "PASS: All 427 interview-only participants "
    "have WTMEC2YR=0 exactly."
)


# ============================================================
# 15. VALIDATE INDFMPIR
# ============================================================

#
# CDC:
#
# 0 - 4.99 = 7,699
# 5        = 1,220
# Missing  = 1,052
#
#
# Attached file also contains exactly:
#
#     92 values = 0
#
# These zeros are legitimate poverty-ratio values.
#

pir_below_5 = int(
    (
        df["INDFMPIR"]
        .ge(0)
        &
        df["INDFMPIR"]
        .lt(5)
    )
    .sum()
)


pir_5 = int(
    df["INDFMPIR"]
    .eq(5)
    .sum()
)


pir_missing = int(
    df["INDFMPIR"]
    .isna()
    .sum()
)


pir_zero = int(
    df["INDFMPIR"]
    .eq(0)
    .sum()
)


print(
    "\n--- INDFMPIR ---"
)

print(
    "0 to <5:",
    pir_below_5
)

print(
    "5:",
    pir_5
)

print(
    "Missing:",
    pir_missing
)

print(
    "Exact zeros:",
    pir_zero
)


if pir_below_5 != 7699:

    raise ValueError(
        "Unexpected INDFMPIR <5 count."
    )


if pir_5 != 1220:

    raise ValueError(
        "Unexpected INDFMPIR=5 count."
    )


if pir_missing != 1052:

    raise ValueError(
        "Unexpected INDFMPIR missing count."
    )


if pir_zero != 92:

    raise ValueError(
        "Unexpected INDFMPIR zero count."
    )


print(
    "PASS: INDFMPIR structure validated."
)


# ============================================================
# 16. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(
        seqn_values
    ),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 17. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert 44 fields to nullable Int64.
#
#
# DO NOT convert:
#
#     WTINT2YR
#     WTMEC2YR
#     INDFMPIR
#
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(
                values
            ),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 44 whole-number/code fields "
    "converted to nullable Int64."
)

print(
    "PASS: WTINT2YR, WTMEC2YR and INDFMPIR "
    "remain floating point."
)


# ============================================================
# 18. CDC CHECK - SDDSRVYR
# ============================================================

#
# 9 = NHANES 2015-2016
#

if (
    df["SDDSRVYR"]
    .value_counts()
    .to_dict()
    !=
    {
        9: 9971
    }
):

    raise ValueError(
        "SDDSRVYR differs from expected."
    )


print(
    "PASS: SDDSRVYR=9 for all 9,971 records."
)


# ============================================================
# 19. CDC CHECK - RIDSTATR
# ============================================================

ridstatr_counts = (
    df["RIDSTATR"]
    .value_counts()
    .sort_index()
    .to_dict()
)


EXPECTED_RIDSTATR = {
    1: 427,
    2: 9544
}


if ridstatr_counts != EXPECTED_RIDSTATR:

    raise ValueError(
        "RIDSTATR frequencies differ.\n"
        f"Expected: {EXPECTED_RIDSTATR}\n"
        f"Actual:   {ridstatr_counts}"
    )


print(
    "PASS: RIDSTATR frequencies validated."
)


# ============================================================
# 20. CDC CHECK - RIAGENDR
# ============================================================

gender_counts = (
    df["RIAGENDR"]
    .value_counts()
    .sort_index()
    .to_dict()
)


EXPECTED_GENDER = {
    1: 4892,
    2: 5079
}


if gender_counts != EXPECTED_GENDER:

    raise ValueError(
        "RIAGENDR frequencies differ."
    )


print(
    "PASS: RIAGENDR frequencies validated."
)


# ============================================================
# 21. CDC CHECK - RIDAGEYR
# ============================================================

#
# CDC:
#
# 0-79 = 9,595
# 80   =   376
#
# Attached XPT has exactly 396 age-zero infants.
#

ridageyr_0_79 = int(
    df["RIDAGEYR"]
    .between(
        0,
        79
    )
    .sum()
)


ridageyr_80 = int(
    df["RIDAGEYR"]
    .eq(80)
    .sum()
)


ridageyr_zero = int(
    df["RIDAGEYR"]
    .eq(0)
    .sum()
)


if ridageyr_0_79 != 9595:

    raise ValueError(
        "RIDAGEYR 0-79 count differs."
    )


if ridageyr_80 != 376:

    raise ValueError(
        "RIDAGEYR age-80 count differs."
    )


if ridageyr_zero != 396:

    raise ValueError(
        "RIDAGEYR age-zero count differs."
    )


print(
    "PASS: RIDAGEYR range/zero counts validated."
)


# ============================================================
# 22. CDC CHECK - RIDAGEMN
# ============================================================

#
# Age in months at screening for young children.
#
# 0-24 = 695
# Missing = 9,276
#
# 31 are legitimate zero-month values.
#

ridagemn_range = int(
    df["RIDAGEMN"]
    .between(
        0,
        24
    )
    .sum()
)


ridagemn_zero = int(
    df["RIDAGEMN"]
    .eq(0)
    .sum()
)


ridagemn_missing = int(
    df["RIDAGEMN"]
    .isna()
    .sum()
)


if ridagemn_range != 695:

    raise ValueError(
        "RIDAGEMN 0-24 count differs."
    )


if ridagemn_zero != 31:

    raise ValueError(
        "RIDAGEMN zero count differs."
    )


if ridagemn_missing != 9276:

    raise ValueError(
        "RIDAGEMN missing count differs."
    )


print(
    "PASS: RIDAGEMN validated."
)


# ============================================================
# 23. CDC CHECK - RIDEXAGM
# ============================================================

#
# Age in months at examination.
#
# Publicly released for age <240 months.
#
# Attached:
#
#     0-239 = 4,060
#     zero  = 10
#     missing = 5,911
#

ridexagm_range = int(
    df["RIDEXAGM"]
    .between(
        0,
        239
    )
    .sum()
)


ridexagm_zero = int(
    df["RIDEXAGM"]
    .eq(0)
    .sum()
)


ridexagm_missing = int(
    df["RIDEXAGM"]
    .isna()
    .sum()
)


if ridexagm_range != 4060:

    raise ValueError(
        "RIDEXAGM range count differs."
    )


if ridexagm_zero != 10:

    raise ValueError(
        "RIDEXAGM zero count differs."
    )


if ridexagm_missing != 5911:

    raise ValueError(
        "RIDEXAGM missing count differs."
    )


print(
    "PASS: RIDEXAGM validated."
)


# ============================================================
# 24. CDC CHECK - DMDEDUC3
# ============================================================

#
# Education for participants age 6-19.
#
# 0 means:
#
#     Never attended / kindergarten only.
#
# CDC count = 239.
#

DMDEDUC3_EXPECTED = {
    0: 239,
    1: 252,
    2: 237,
    3: 220,
    4: 217,
    5: 205,
    6: 180,
    7: 179,
    8: 210,
    9: 159,
    10: 160,
    11: 153,
    12: 34,
    13: 124,
    14: 7,
    15: 58,
    55: 3,
    66: 9,
    99: 1
}


actual_dmdeduc3 = (
    df["DMDEDUC3"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_dmdeduc3 != DMDEDUC3_EXPECTED:

    raise ValueError(
        "DMDEDUC3 frequencies differ.\n"
        f"Expected: {DMDEDUC3_EXPECTED}\n"
        f"Actual:   {actual_dmdeduc3}"
    )


if int(
    df["DMDEDUC3"]
    .isna()
    .sum()
) != 7324:

    raise ValueError(
        "DMDEDUC3 missing count differs."
    )


print(
    "PASS: DMDEDUC3 matches CDC exactly."
)


# ============================================================
# 25. CDC CHECK - HOUSEHOLD COMPOSITION
# ============================================================

#
# DMDHHSZA:
#     children <=5
#
# DMDHHSZB:
#     children 6-17
#
# DMDHHSZE:
#     adults >=60
#

EXPECTED_HHSZA = {
    0: 6298,
    1: 2147,
    2: 1199,
    3: 327
}


EXPECTED_HHSZB = {
    0: 4715,
    1: 1990,
    2: 1833,
    3: 822,
    4: 611
}


EXPECTED_HHSZE = {
    0: 7151,
    1: 1663,
    2: 1099,
    3: 58
}


if (
    df["DMDHHSZA"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    EXPECTED_HHSZA
):

    raise ValueError(
        "DMDHHSZA frequencies differ."
    )


if (
    df["DMDHHSZB"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    EXPECTED_HHSZB
):

    raise ValueError(
        "DMDHHSZB frequencies differ."
    )


if (
    df["DMDHHSZE"]
    .value_counts()
    .sort_index()
    .to_dict()
    !=
    EXPECTED_HHSZE
):

    raise ValueError(
        "DMDHHSZE frequencies differ."
    )


print(
    "PASS: Household-composition variables "
    "match CDC exactly."
)


# ============================================================
# 26. CDC CHECK - HOUSEHOLD / FAMILY SIZE
# ============================================================

#
# DMDHHSIZ and DMDFMSIZ are coded 1-7,
# with 7 meaning 7 or more.
#

if int(
    df["DMDHHSIZ"]
    .between(
        1,
        7
    )
    .sum()
) != 9971:

    raise ValueError(
        "DMDHHSIZ contains unexpected values."
    )


if int(
    df["DMDFMSIZ"]
    .between(
        1,
        7
    )
    .sum()
) != 9971:

    raise ValueError(
        "DMDFMSIZ contains unexpected values."
    )


print(
    "PASS: Household/family size range 1-7 validated."
)


# ============================================================
# 27. CDC CHECK - SURVEY DESIGN VARIABLES
# ============================================================

#
# 15 masked strata:
#
#     119-133
#
# Each PSU:
#
#     1 or 2
#

if int(
    df["SDMVSTRA"]
    .between(
        119,
        133
    )
    .sum()
) != 9971:

    raise ValueError(
        "Unexpected SDMVSTRA values."
    )


if int(
    df["SDMVSTRA"]
    .nunique()
) != 15:

    raise ValueError(
        "Expected 15 masked strata."
    )


if set(
    df["SDMVPSU"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
) != {
    1,
    2
}:

    raise ValueError(
        "Unexpected SDMVPSU values."
    )


print(
    "PASS: Survey design variables validated."
)


# ============================================================
# 28. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Before correction:
#
#     ordinary zeros = 0
#
# After correction the ONLY zeros should be exactly
# these 19,359 validated records.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO PATTERN ---"
)


print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from validated "
        "DEMO_I correction pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 19359:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 19,359 validated restorations."
)


# ============================================================
# 29. NUMERIC FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY allowed changes are:
#
#     validated tiny -> 0
#
# No other numeric value may change.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            fidelity_failures
        )
    )


print(
    "PASS: All 47 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 30. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
#
# This is especially important for:
#
#     WTINT2YR
#     WTMEC2YR
#     INDFMPIR
#
# which must retain their released decimal values.
#
# Same principle applies to your Dietary decimal fields.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 31. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 32. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: All 47 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 33. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 34. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 35. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(
            TINY_VALUE
        )
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 36. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DEMO_I contains genuine decimal fields.
#
# Therefore DO NOT globally search every column
# for ".0".
#
# Check ONLY the 44 proven integer/code fields.
#

print(
    "\n--- INTEGER FIELD .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' in whole-number fields:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 44 whole-number/code fields."
)


# ============================================================
# 37. DECIMAL .0 TOKENS ARE LEGITIMATE
# ============================================================

#
# INFORMATIONAL ONLY.
#
# These are GENUINE decimal columns.
#
# A value ending in ".0" is NOT an error.
#
# Examples:
#
# WTMEC2YR
#     0.0 = legitimate not-MEC-examined sample weight
#
# INDFMPIR
#     0.0
#     1.0
#     5.0
#
# can all be legitimate values in a continuous field.
#

print(
    "\n--- LEGITIMATE DECIMAL .0 TOKENS ---"
)


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col]
            .ne("")
            &
            raw_csv_df[col]
            .str.endswith(".0")
        )
        .sum()
    )


    decimal_dot_zero_counts[col] = count


    print(
        f"{col}:",
        f"{count:,}"
    )


print(
    "PASS: Decimal '.0' tokens are left AS IS."
)


# ============================================================
# 38. EXACT DECIMAL FIDELITY
# ============================================================

#
# Require bit-for-bit numeric round-trip equality
# after CSV read with float_precision='round_trip'.
#

print(
    "\n--- EXACT DECIMAL FIDELITY ---"
)


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        raise ValueError(
            f"{col} decimal fidelity failed."
        )


    print(
        f"PASS: {col} preserved exactly."
    )


# ============================================================
# 39. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV zero pattern exactly "
    "matches the 19,359 validated restorations."
)


# ============================================================
# 40. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 41. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 42. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "DEMO_I CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Demographics"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "RIDAGEYR age-zero restorations:",
    EXPECTED_TINY_COUNTS["RIDAGEYR"]
)


print(
    "RIDAGEMN zero-month restorations:",
    EXPECTED_TINY_COUNTS["RIDAGEMN"]
)


print(
    "RIDEXAGM zero-month restorations:",
    EXPECTED_TINY_COUNTS["RIDEXAGM"]
)


print(
    "DMDEDUC3 zero restorations:",
    EXPECTED_TINY_COUNTS["DMDEDUC3"]
)


print(
    "DMDHHSZA zero restorations:",
    EXPECTED_TINY_COUNTS["DMDHHSZA"]
)


print(
    "DMDHHSZB zero restorations:",
    EXPECTED_TINY_COUNTS["DMDHHSZB"]
)


print(
    "DMDHHSZE zero restorations:",
    EXPECTED_TINY_COUNTS["DMDHHSZE"]
)


print(
    "WTMEC2YR zero restorations:",
    EXPECTED_TINY_COUNTS["WTMEC2YR"]
)


print(
    "INDFMPIR zero restorations:",
    EXPECTED_TINY_COUNTS["INDFMPIR"]
)


print(
    "Interview-only participants:",
    int(
        df["RIDSTATR"]
        .eq(1)
        .sum()
    )
)


print(
    "MEC examined participants:",
    int(
        df["RIDSTATR"]
        .eq(2)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)


print(
    "Decimal '.0' values: "
    "ALLOWED / PRESERVED"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2015-2016 DEMO_I XPT -> CSV
Rows: 9,971
Columns: 47
PASS: Structure = 9,971 rows x 47 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['WTINT2YR', 'WTMEC2YR', 'INDFMPIR']
Whole-number/code fields: 44

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 19,359

Tiny values by variable:
DMDHHSZE    7151
DMDHHSZA    6298
DMDHHSZB    4715
WTMEC2YR     427
RIDAGEYR     396
DMDEDUC3     239
INDFMPIR      92
RIDAGEMN      31
RIDEXAGM      10

PASS: Exact 19,359-value tiny-zero pattern across 9 variables validated.

--- VALIDATED ZERO RESTORATION ---
RIDAGEYR: 396 restored to 0
RIDAGEMN: 31 restored to 0
RIDEXAGM: 10 restored to 0
DMDEDUC3: 239 restored to 0
DMDHHSZA: 6,298 restored to 0
DMDHHSZB: 4,715 restored to 0
DMDHHSZE: 7,151 restored to 0
WTMEC2YR: 427 restored to 0
INDFMPIR: 92 restored to 0

PASS: 19,359 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTIO